In [ ]:
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

import scipy.sparse as sp
from collections import defaultdict
from os.path import join

os.environ['R_HOME'] = '/home/yanxh/anaconda3/envs/R4.3/lib/R'
os.environ['R_USER'] = '/home/yanxh/anaconda3/envs/R4.3/lib/python3.8/site-packages/rpy2'

In [ ]:
def run_mclust(adata, num_cluster, used_obsm='emb', modelNames='EEE', random_seed=2020):
    np.random.seed(random_seed)
    import rpy2.robjects as robjects
    robjects.r.library("mclust")

    import rpy2.robjects.numpy2ri
    rpy2.robjects.numpy2ri.activate()
    r_random_seed = robjects.r['set.seed']
    r_random_seed(random_seed)
    rmclust = robjects.r['Mclust']

    res = rmclust(rpy2.robjects.numpy2ri.numpy2rpy(adata.obsm[used_obsm]), num_cluster, modelNames)
    mclust_res = np.array(res[-2])

    return mclust_res

In [ ]:
def run_comm_detection(adata, num_cluster, method='louvain', used_obsm='emb'):
    from scanpy.tl import leiden, louvain
    from sklearn.neighbors import NearestNeighbors

    assert method in ['louvain', 'leiden'], "method must be 'louvain' or 'leiden'"

    sc.pp.neighbors(adata, use_rep=used_obsm)

    best_labels = None
    best_diff = float('inf')
    resolutions = np.concatenate([np.arange(0.1, 2.1, 0.1), np.arange(0.01, 2.01, 0.01)])
    seen = set()

    for res in resolutions:
        if round(res, 2) in seen:
            continue
        seen.add(round(res, 2))
        key = '__temp_clust__'
        if method == 'louvain':
            louvain(adata, resolution=res, key_added=key)
        else:
            leiden(adata, resolution=res, key_added=key)
        labels = adata.obs[key]
        n = len(np.unique(labels))
        if n == num_cluster:
            adata.obs['cluster'] = labels.astype(str)
            return adata.obs['cluster'].to_list()
        diff = abs(n - num_cluster)
        if diff < best_diff:
            best_diff = diff
            best_labels = labels.astype(str)

    adata.obs['cluster'] = best_labels
    return adata.obs['cluster'].to_list()

In [ ]:
clusters_map = { 
    "Misar-E13-E15-E18": 14, 
    "Misar-E15-E18-E18": 14, 
    "Lymph-S1-S2-S3":10, 
    "Lymph-S1-S2-S2":10, 
    "Tonsil-S1-S2-S3":4, 
    "Tonsil-S1-S2-S2":4, 
    "MB-5M":15, 
    "MB-H3K4me3":13,
    "DBiT-RnaAtac-Atac":15,
    "Misar-Stereo":18,
    "Mux-VisiumHD-HE":18,
    "Embryo":13
}

methods_map = {
    'BANKSY': 'leiden',
    'Leiden-H3K27ac': 'leiden',
    'CLUE': 'leiden',
    'Leiden-Protein': 'leiden',
    'Leiden-H3K27me3': 'leiden',
    'StabMap': 'leiden',
    'SpaMosaic(non-spatial)': 'mclust',
    'UINMF': 'louvain',
    'Leiden-ATAC': 'leiden',
    'scMoMaT': 'leiden',
    'Leiden-H3K4me3': 'leiden',
    'SpaMosaic': 'mclust',
    'MIDAS': 'louvain',
    'Cobolt': 'louvain',
    'MultiVI': 'leiden',
    'Leiden-RNA': 'leiden'
}

In [ ]:
data_path = "../../results/embeddings"
save_path = '../../results/clusters'

methods = os.listdir(data_path)
for method, cluster_method in methods_map.items():
    _dir = join(data_path, method)
    datasets = os.listdir(_dir)
    for ds in datasets:
        print(method, ds)
        df_emb = pd.read_csv(join(_dir, ds, 'df_emb.csv'), index_col=0)
        adata = sc.AnnData(df_emb.values)
        adata.obs_names = df_emb.index
        adata.obsm['emb'] = df_emb.values
        
        if cluster_method=='mclust':
            clusters = run_mclust(adata, clusters_map[ds], used_obsm='emb')
        else:
            clusters = run_comm_detection(adata, clusters_map[ds], method=cluster_method, used_obsm='emb')
            
        os.makedirs(join(save_path, method, ds), exist_ok=True)
        pd.DataFrame(clusters, index=df_emb.index, columns=['cluster']).to_csv(join(save_path, method, ds, 'df_cluster.csv'))